# Campylobacter analysis figures — Figure 1 (automatic vs semi-automatic scoring) and Figure 2 (negotiation outcomes and argument influence)

Builds the two combined manuscript figures from the pipeline outputs. Nothing is re-sampled and no API is called.

| Input (inside `PROJECT_DIR`) | Used for |
|---|---|
| `analysis_outputs/` | Fig. 1a (participant scores vs automatic intervals); package utilities for reconstructing the automatic profile |
| `perturbation_outputs/` | Fig. 1b–c (automatic pipeline adaptation); adapted score profile for Fig. 2a |
| `participant_perturbation_outputs/` | Fig. 1b–c (semi-automatic pipeline adaptation) |
| `adapted_bayesian_negotiation_outputs/` | Fig. 2a (final deals of the adapted Bayesian negotiations) |
| `trace-*.json` (LangSmith export of one run) | Fig. 2b (argument influence) |

Figures are sized for a full page width (about 7 in) and written as PNG (300 dpi) and PDF to `figure_outputs/`.

## 1 — Paths and shared style

In [ ]:
from pathlib import Path
import json, re, itertools, math
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.transforms import blended_transform_factory
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

PROJECT_DIR = Path('/content/drive/MyDrive/MicRisk_agents_SDK/Campy_case_Bayesian')
if not PROJECT_DIR.exists():
    PROJECT_DIR = Path.cwd()
AN, AU, PA, RUNS = (PROJECT_DIR / d for d in ['analysis_outputs', 'perturbation_outputs',
                                               'participant_perturbation_outputs', 'adapted_bayesian_negotiation_outputs'])
TRACE_FILE = next(PROJECT_DIR.glob('trace-*.json'))
OUT = PROJECT_DIR / 'figure_outputs'; OUT.mkdir(exist_ok=True)
THRESHOLD = 65

# One style for both figures (the Fig. 4 style): DejaVu Sans, bold panel titles, black-edged bars,
# steel blue / coral / grey. Sizes are chosen for a 14-inch canvas printed at ~7 inches (≈ 8–9 pt).
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 15, 'axes.titlesize': 17, 'axes.titleweight': 'bold',
                     'axes.labelsize': 15, 'xtick.labelsize': 13.5, 'ytick.labelsize': 13.5, 'legend.fontsize': 13.5,
                     'axes.spines.top': False, 'axes.spines.right': False, 'savefig.dpi': 300})
FS = dict(letter=24, title=17, tick=13.5, small=12.5, ann=14, leg=13.5)
BLUE, CORAL, GREY, PALE, INK = 'steelblue', 'coral', 'grey', '#9ebbd0', '#222222'
INSIDE, OUTSIDE = '#2ca02c', '#d62728'
DIVERGING = LinearSegmentedColormap.from_list('div', ['#1f4e79', BLUE, '#f2f2f2', CORAL, '#b03a2e'])
SHORT = {'Consumers': 'Cons', 'Food industry': 'FI', 'Food Safety Authorities': 'FSA', 'Environmental NGOs': 'NGOs'}

def letter(fig_or_sub, text, x=0.0, y=1.0):
    fig_or_sub.text(x, y, text, fontsize=FS['letter'], fontweight='bold', va='top', ha='left')

def clean(ax, keep_left=True):
    for s in ('top', 'right') + (() if keep_left else ('left',)):
        ax.spines[s].set_visible(False)

def save(fig, name):
    for ext in ('png', 'pdf'):
        fig.savefig(OUT / f'{name}.{ext}', bbox_inches='tight', dpi=300)
    print('Saved', OUT / f'{name}.png')

## 2 — Data for Figure 1

The minimum-transfer search (Fig. 1c) enumerates every valid integer re-weighting with at most `CAP` transferred points per stakeholder, using the same rules as Step 8 (100-point budgets, order-preserving integer re-quantisation, frozen zero-weight issues). It takes 3–5 min and is cached in `figure_outputs/`.

In [ ]:
def integer_rescale(source, target_maximum):
    target_maximum = int(target_maximum); old_maximum = int(max(source.values()))
    if old_maximum == 0:
        return {o: 0 for o in source}
    levels = sorted(set(int(x) for x in source.values()))
    mapped = [int(round(l * target_maximum / old_maximum)) for l in levels]
    if levels[0] == 0: mapped[0] = 0
    mapped[-1] = target_maximum
    for i in range(1, len(mapped)): mapped[i] = max(mapped[i], mapped[i - 1] + 1)
    mapped[-1] = target_maximum
    for i in range(len(mapped) - 2, -1, -1): mapped[i] = min(mapped[i], mapped[i + 1] - 1)
    if levels[0] == 0: mapped[0] = 0
    lookup = dict(zip(levels, mapped))
    return {o: int(lookup[int(v)]) for o, v in source.items()}

def reconstruct_profile(package_csv, weights_csv):
    # additive model: within-issue differences come from the package utilities; the best option equals the issue weight
    pk = pd.read_csv(package_csv); w = pd.read_csv(weights_csv)
    names = list(dict.fromkeys(w.stakeholder)); issues = sorted(w.issue.unique())
    parts = pk.package.str.split(', ', expand=True); parts.columns = issues
    prof = {}
    for n in names:
        prof[n] = {}
        for i in issues:
            mask = np.all([parts[j] == parts[j].iloc[0] for j in issues if j != i], axis=0)
            sub = pk[mask].set_index(parts[mask][i])[n]
            weight = int(w[(w.stakeholder == n) & (w.issue == i)].original_weight.iloc[0])
            prof[n][i] = {o: int(round(weight - (sub.max() - v))) for o, v in sub.sort_index().items()}
        u = sum(parts[i].map(prof[n][i]) for i in issues)
        assert np.array_equal(u.to_numpy(), pk[n].round().astype(int).to_numpy()), f'reconstruction failed for {n}'
    return prof

def weight_vectors(tables, cap):
    issues = sorted(tables)
    w0 = np.array([max(tables[i].values()) for i in issues]); active = np.flatnonzero(w0 > 0)
    mins = np.array([max(1, len(set(tables[i].values())) - 1) if w0[k] > 0 else 0 for k, i in enumerate(issues)])
    lo, hi = np.maximum(mins, w0 - cap), w0 + cap
    grids = [np.arange(lo[k], hi[k] + 1) for k in active[:-1]]
    prefix = np.array(np.meshgrid(*grids, indexing='ij')).reshape(len(grids), -1).T
    last = w0[active].sum() - prefix.sum(1); keep = (last >= lo[active[-1]]) & (last <= hi[active[-1]])
    W = np.tile(w0, (int(keep.sum()), 1)); W[:, active[:-1]] = prefix[keep]; W[:, active[-1]] = last[keep]
    t = np.abs(W - w0).sum(1) // 2
    return issues, w0, W[t <= cap], t[t <= cap]

def min_transfer_per_package(tables, target=66, cap=30, chunk=20000):
    issues, w0, W, transfer = weight_vectors(tables, cap)
    opts = {i: sorted(tables[i]) for i in issues}
    packages = list(itertools.product(*(opts[i] for i in issues)))
    lookup = []
    for k, i in enumerate(issues):
        arr = np.zeros((W[:, k].max() + 1, len(opts[i])), dtype=np.int16)
        for v in np.unique(W[:, k]):
            t = integer_rescale(tables[i], int(v)); arr[v] = [t[o] for o in opts[i]]
        lookup.append(arr)
    idx = [np.array([opts[i].index(p[k]) for p in packages]) for k, i in enumerate(issues)]
    best = np.full(len(packages), np.inf)
    for s in range(0, len(W), chunk):
        Wc = W[s:s + chunk]; val = np.zeros((len(Wc), len(packages)), dtype=np.int32)
        for k in range(len(issues)): val += lookup[k][Wc[:, k][:, None], idx[k][None, :]]
        best = np.minimum(best, np.where(val >= target, transfer[s:s + chunk, None], np.inf).min(0))
    return [', '.join(p) for p in packages], best

def first_nbs_quality(profile, package, target=66, cap=30):
    # rebuild the minimum-transfer landscape for `package` and return its NBS (ties: transfer, max change, sumsq, weights)
    chosen = package.split(', '); adapted = {}
    for n, tables in profile.items():
        issues, w0, W, t = weight_vectors(tables, cap)
        d = W - w0; value = np.zeros(len(W), dtype=int)
        for k, i in enumerate(issues):
            col = {int(v): integer_rescale(tables[i], int(v))[chosen[k]] for v in np.unique(W[:, k])}
            value += np.vectorize(col.get)(W[:, k])
        ok = value >= target; W, d, t = W[ok], d[ok], t[ok]
        r = np.lexsort([W[:, j] for j in reversed(range(W.shape[1]))] + [np.square(d).sum(1), np.abs(d).max(1), t])[0]
        adapted[n] = {i: integer_rescale(tables[i], int(W[r, k])) for k, i in enumerate(issues)}
    names = list(adapted); issues = sorted(adapted[names[0]])
    pk = list(itertools.product(*(sorted(adapted[names[0]][i]) for i in issues)))
    U = np.array([[sum(adapted[n][i][o] for i, o in zip(issues, p)) for n in names] for p in pk])
    nash = np.where(np.all(U > THRESHOLD, 1), np.prod(U - THRESHOLD, 1), -1)
    k = max(np.flatnonzero(nash == nash.max()), key=lambda j: U[j].sum())
    return {'package': ', '.join(pk[k]), 'total': int(U[k].sum()), 'nash': int(nash[k])}

coverage = pd.read_csv(AN / 'participant_score_interval_coverage.csv')
d_auto = pd.read_csv(AU / 'smallest_genuine_nbs_weight_changes.csv')
d_part = pd.read_csv(PA / 'smallest_genuine_nbs_weight_changes.csv')
res_auto = json.loads((AU / 'balanced_perturbation_results.json').read_text())
res_part = json.loads((PA / 'balanced_perturbation_results.json').read_text())
STAKEHOLDERS = list(dict.fromkeys(d_auto.stakeholder))
ISSUES = sorted(d_auto.issue.unique())

auto_profile = reconstruct_profile(AN / 'equilibrium_packages_median_representative.csv', AU / 'smallest_genuine_nbs_weight_changes.csv')
part_profile = {n: {i: {o: int(v) for o, v in t.items()} for i, t in r['utility_table'].items()}
                for n, r in json.loads((PA / 'participant_original_profile.json').read_text())['stakeholders'].items()}
CAP = 30
transfer = {}
for label, prof in [('Automatic', auto_profile), ('Semi-automatic', part_profile)]:
    cache = OUT / f'min_total_transfer_{label.lower()}_cap{CAP}.csv'
    if cache.exists():
        transfer[label] = pd.read_csv(cache, index_col=0); continue
    cols = {}
    for n in STAKEHOLDERS:
        packages, cols[n] = min_transfer_per_package(prof[n], cap=CAP); print(label, n, 'done')
    df = pd.DataFrame(cols, index=packages); df['total'] = df.sum(axis=1); df.to_csv(cache); transfer[label] = df
first = {}
for label, prof in [('Automatic', auto_profile), ('Semi-automatic', part_profile)]:
    best = transfer[label].total.idxmin()
    first[label] = dict(points=int(transfer[label].total.min()), **first_nbs_quality(prof, best, cap=CAP))
print(first)

## 3 — Figure 1: automatic vs semi-automatic scoring

(a) Participant scores against the automatic p05–p95 intervals. (b) Smallest integer adaptation creating a genuine NBS at 65, side by side. (c) Packages that can be made acceptable to all parties within a total-transfer budget, with the quality of the first NBS.

In [ ]:
fig = plt.figure(figsize=(14, 13.2))
top, bottom = fig.subfigures(2, 1, height_ratios=[1.12, 1], hspace=0.04)

# ---- (a) participant scores vs automatic intervals
axes = top.subplots(2, 2, sharey=True, gridspec_kw={'hspace': 0.42, 'wspace': 0.05})
outline = [pe.Stroke(linewidth=4.5, foreground='black'), pe.Normal()]
ymax = max(45, float(coverage[['automatic_maximum', 'participant_score']].to_numpy().max()) + 4)
for ax, name in zip(axes.flat, list(dict.fromkeys(coverage.stakeholder))):
    rows = coverage[coverage.stakeholder == name].reset_index(drop=True); x = np.arange(len(rows))
    ax.vlines(x, rows.automatic_minimum, rows.automatic_maximum, color='#cfd6df', linewidth=4, zorder=1)
    ax.vlines(x, rows.automatic_p05, rows.automatic_p95, color=BLUE, linewidth=8, zorder=2)
    ax.scatter(x, rows.automatic_median, s=30, color='white', edgecolor='black', linewidth=1.1, zorder=3)
    ins = rows.inside_central_90_interval.to_numpy(bool)
    for mask, color in [(ins, INSIDE), (~ins, OUTSIDE)]:
        ax.scatter(x[mask], rows.participant_score[mask], marker='x', s=80, linewidth=2.4, color=color, zorder=5, path_effects=outline)
    for b in np.flatnonzero(rows.issue.to_numpy()[1:] != rows.issue.to_numpy()[:-1]) + 0.5:
        ax.axvline(b, color='#d0d0d0', linewidth=1, zorder=0)
    ax.set_xticks(x, rows.option, fontsize=FS['small'], rotation=90)
    ax.set_xlim(-0.7, len(rows) - 0.3); ax.set_ylim(-3, ymax)
    ax.set_title(f'{name}: {int(ins.sum())}/{len(rows)} inside', loc='left', fontsize=FS['title'] - 1)
    ax.grid(axis='y', linestyle='--', alpha=0.6); ax.set_axisbelow(True); clean(ax)
for ax in axes[:, 0]: ax.set_ylabel('Option score (points)')
inside_n, total_n = int(coverage.inside_central_90_interval.sum()), len(coverage)
top.legend(handles=[Line2D([0], [0], color='#cfd6df', lw=4, label='Automatic: simulated min–max'),
                    Line2D([0], [0], color=BLUE, lw=8, label='Automatic: p05–p95'),
                    Line2D([0], [0], marker='o', ls='', mfc='white', mec='black', label='Automatic: median'),
                    Line2D([0], [0], marker='x', ls='', color=INSIDE, ms=10, mew=2.4, path_effects=outline, label='Participant: inside p05–p95'),
                    Line2D([0], [0], marker='x', ls='', color=OUTSIDE, ms=10, mew=2.4, path_effects=outline, label='Participant: outside p05–p95')],
           loc='lower center', ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.045), columnspacing=1.6)
top.subplots_adjust(left=0.07, right=0.99, top=0.85, bottom=0.17)
top.text(0.045, 0.975, f'Participant scores vs automatic score intervals ({inside_n}/{total_n} inside the central 90%)',
         fontsize=FS['title'], fontweight='bold', va='top')
letter(top, 'a', 0.0, 0.985)

# ---- (b) heatmaps and (c) reachability curve
bl, br = bottom.subfigures(1, 2, width_ratios=[1.25, 1], wspace=0.02)
hm = bl.subplots(1, 2, sharey=True, gridspec_kw={'wspace': 0.06})
limit = int(max(d_auto.change.abs().max(), d_part.change.abs().max()))
for ax, d, res, label in [(hm[0], d_auto, res_auto, 'Automatic'), (hm[1], d_part, res_part, 'Semi-automatic')]:
    ch = d.pivot(index='stakeholder', columns='issue', values='change').reindex(index=STAKEHOLDERS, columns=ISSUES)
    w0 = d.pivot(index='stakeholder', columns='issue', values='original_weight').reindex(index=STAKEHOLDERS, columns=ISSUES)
    im = ax.imshow(ch.to_numpy(float), cmap=DIVERGING, vmin=-limit, vmax=limit, aspect='auto')
    for i in range(len(STAKEHOLDERS)):
        for j in range(len(ISSUES)):
            if w0.iat[i, j] == 0:
                ax.add_patch(plt.Rectangle((j - .5, i - .5), 1, 1, facecolor='white', edgecolor='#b5b5b5', hatch='///', lw=0)); continue
            v = int(ch.iat[i, j])
            ax.text(j, i, f'{v:+d}' if v else '0', ha='center', va='center', fontsize=FS['ann'],
                    fontweight='bold' if v else 'normal', color='white' if abs(v) > 0.6 * limit else INK)
    ax.set_xticks(range(len(ISSUES)), ISSUES); ax.set_yticks(range(len(STAKEHOLDERS)), [SHORT[s] for s in STAKEHOLDERS])
    ax.set_xticks(np.arange(-.5, len(ISSUES)), minor=True); ax.set_yticks(np.arange(-.5, len(STAKEHOLDERS)), minor=True)
    ax.grid(which='minor', color='white', linewidth=2.5); ax.tick_params(which='both', length=0)
    for s in ax.spines.values(): s.set_visible(False)
    sm = res['smallest_practical_genuine_nbs_profile']
    ax.set_title(f"{label}\nNBS {sm['equilibrium']['benchmark_package']}", fontsize=FS['title'] - 2)
    ax.set_xlabel('Issue')
bl.subplots_adjust(left=0.12, right=0.98, top=0.78, bottom=0.3)
cax = bl.add_axes([0.2, 0.075, 0.7, 0.035])
cb = bl.colorbar(im, cax=cax, orientation='horizontal')
cb.set_label('Issue-weight change (points); hatched = zero-weight issue'); cb.outline.set_visible(False)
letter(bl, 'b', 0.0, 0.99)
bl.text(0.12, 0.985, 'Smallest adaptation creating an NBS at 65', fontsize=FS['title'], fontweight='bold', va='top')

ax = br.subplots()
budgets = np.arange(0, CAP + 1)
for label, color, ytxt in [('Automatic', BLUE, 0.97), ('Semi-automatic', CORAL, 0.69)]:
    totals = transfer[label].total.to_numpy()
    ax.step(budgets, [(totals <= b).sum() for b in budgets], where='post', color=color, lw=3, label=label)
    f = first[label]
    ax.plot(f['points'], 1, 'o', ms=11, color=color, mec='black', mew=1.3, zorder=5)
    ax.text(0.04, ytxt, f"{label}: first NBS after {f['points']} points\n{f['package']}\nTotal utility {f['total']} · Nash product {f['nash']}",
            transform=ax.transAxes, fontsize=FS['small'], va='top', linespacing=1.35,
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor=color, linewidth=2))
ax.set_xlim(0, CAP + 0.5); ax.set_ylim(0, max(11.5, ax.get_ylim()[1] * 1.4))
ax.set_xlabel('Total issue-weight points transferred'); ax.set_ylabel('Packages acceptable to all (≥ 66)')
ax.grid(linestyle='--', alpha=0.6); ax.set_axisbelow(True); clean(ax)
ax.legend(frameon=False, loc='center left', bbox_to_anchor=(0.02, 0.3))
br.subplots_adjust(left=0.17, right=0.97, top=0.78, bottom=0.21)
letter(br, 'c', 0.0, 0.99)
br.text(0.1, 0.985, 'Distance to an NBS', fontsize=FS['title'], fontweight='bold', va='top')
save(fig, 'Figure1_automatic_vs_semiautomatic')
plt.show()

## 4 — Data for Figure 2

(a) Every final deal of the completed adapted Bayesian negotiations, compared with the NBS of the adapted profile. Unfinished runs (no `run_summary.json`) are skipped. (b) Public statements turned into evidence in one traced negotiation: the shift in P(accept) for the same deal evaluated just before and after the statement, and the share of the opponent model it ruled out. Agents' private scratchpads are never read.

In [ ]:
tables = res_auto['smallest_practical_genuine_nbs_profile']['utility_table']
adapted_file = PROJECT_DIR / 'adapted_profiles' / 'profile_integer_adapted_smallest.json'
if adapted_file.exists():
    tables = {n: r['utility_table'] for n, r in json.loads(adapted_file.read_text())['stakeholders'].items()}
opts = {i: sorted(tables[STAKEHOLDERS[0]][i], key=lambda o: int(o[1:])) for i in ISSUES}
util = lambda deal: tuple(sum(tables[s][i][o] for i, o in zip(ISSUES, deal.split(', '))) for s in STAKEHOLDERS)
ALL = [', '.join(p) for p in itertools.product(*(opts[i] for i in ISSUES))]
U = np.array([util(p) for p in ALL], float); G = U - THRESHOLD
is_ir = np.all(G >= 0, 1); nashv = np.where(is_ir, np.prod(G, 1), np.nan)
pareto = ~np.array([np.any(np.all(U >= U[k], 1) & np.any(U > U[k], 1)) for k in range(len(U))])
max_nash = np.nanmax(nashv); k_nbs = max(np.flatnonzero(nashv == max_nash), key=lambda k: U[k].sum())
NBS, U_nbs = ALL[k_nbs], U[k_nbs]; IDX = {p: k for k, p in enumerate(ALL)}

deals, skipped = [], []
for folder in sorted(p for p in RUNS.iterdir() if p.is_dir()):
    if not (folder / 'run_summary.json').exists():
        skipped.append(folder.name); continue
    s = json.loads((folder / 'run_summary.json').read_text())
    m = re.search(r'smallest_(.+)_moderator_run_(\d+)', folder.name)
    mod = {n.lower().replace(' ', '_'): n for n in STAKEHOLDERS}.get(m.group(1), m.group(1))
    rep = {x['stakeholder']: x['true_score_evaluation_only'] for x in s['stakeholders']}
    assert all(abs(dict(zip(STAKEHOLDERS, util(s['primary_final_deal'])))[k] - v) < 1e-9 for k, v in rep.items()), folder.name
    agreed = s.get('agreement_reached', s.get('proposal_passes', True))
    blocker = (s.get('main_disagreement') or {}).get('stakeholder', '')
    deals += [{'deal': d, 'moderator': mod, 'agreed': bool(agreed), 'blocker': blocker} for d in s['final_deals']]
deals = pd.DataFrame(deals)
# a rejected final proposal is not an agreement: keep it as its own row, labelled "no agreement"
deals['row'] = np.where(deals.agreed, deals.deal, deals.deal + ' |NA')
g = deals.groupby('row')
f4 = pd.DataFrame({'count': g.size(), 'deal': g.deal.first(), 'agreed': g.agreed.first(),
                   'blockers': g.blocker.apply(lambda b: ', '.join(sorted({SHORT.get(x, x) for x in b if x}))),
                   'mods': g.moderator.apply(lambda m: ', '.join(f'{SHORT[k]} {v}' for k, v in m.value_counts().items()))})
for col, fn in [('ir', lambda d: bool(is_ir[IDX[d]])), ('pareto', lambda d: bool(pareto[IDX[d]])), ('nbs', lambda d: d == NBS),
                ('hamming', lambda d: sum(a != b for a, b in zip(d.split(', '), NBS.split(', ')))),
                ('euclid', lambda d: float(np.sqrt(((U_nbs - U[IDX[d]]) ** 2).sum()))),
                ('manhattan', lambda d: float(np.abs(U_nbs - U[IDX[d]]).sum())),
                ('loss', lambda d: float(abs(nashv[IDX[d]] - max_nash) / max_nash * 100) if is_ir[IDX[d]] else np.nan)]:
    f4[col] = [fn(d) for d in f4.deal]
f4 = f4.reset_index(drop=True).sort_values(['agreed', 'count', 'manhattan'], ascending=[True, True, False]).reset_index(drop=True)
n_runs = len(deals); n_nbs = int(((deals.deal == NBS) & deals.agreed).sum()); n_noagree = int((~deals.agreed).sum())
print(f'{n_runs} final deals from completed runs; NBS {NBS} reached {n_nbs}; skipped: {skipped}')

# ---- trace: evidence events and same-deal before/after pairs
trace = json.loads(TRACE_FILE.read_text())
runs = sorted(trace['runs'], key=lambda r: r['langsmith']['dotted_order'])
calls = []
for r in runs:
    if r['langsmith']['name'] != 'infer_opponent_utility': continue
    o = r.get('outputs') or {}
    if o.get('status') != 'posterior_estimated': continue
    calls.append({'order': r['langsmith']['dotted_order'], 'opp': o['opponent'], 'deal': r['inputs']['candidate_deal'],
                  'p': o['candidate_deal_distribution']['probability_meeting_threshold'], 'surv': o['surviving_fraction'],
                  'ev': tuple(sorted(json.dumps(x, sort_keys=True) for x in o.get('observations_used', [])))})
calls = pd.DataFrame(calls)
surv = {(c.opp, c.ev): c.surv for c in calls.itertuples()}
events = {}
for opp, gg in calls.groupby('opp', sort=False):
    seen = [()]
    for ev in gg.sort_values('order').ev:
        if ev not in seen and set(seen[-1]) <= set(ev):
            new = tuple(sorted(set(ev) - set(seen[-1])))
            events[(opp, new)] = {'opp': opp, 'new': new, 'before': surv.get((opp, seen[-1]), 1.0), 'after': surv[(opp, ev)], 'pairs': []}
            seen.append(ev)
for (opp, deal), gg in calls.groupby(['opp', 'deal'], sort=False):
    gg = gg.sort_values('order')
    for a, b in zip(gg.itertuples(), gg.iloc[1:].itertuples()):
        if a.ev != b.ev and set(a.ev) <= set(b.ev):
            new = tuple(sorted(set(b.ev) - set(a.ev)))
            events.setdefault((opp, new), {'opp': opp, 'new': new, 'before': surv.get((opp, a.ev), 1.0),
                                           'after': surv[(opp, b.ev)], 'pairs': []})['pairs'].append(b.p - a.p)
def describe(e):
    e = json.loads(e)
    if e['type'] == 'comparison': kind = f"{e['preferred']} ≻ {e['less_preferred']}"
    elif e['type'] == 'deal_stance': kind = f"{e['stance']}s {''.join(o for o in e['deal'])}"
    else: kind = e['type']
    return e.get('round'), kind, re.sub(r'</?DEAL>', '', e.get('evidence_quote', '')).strip()
infl = []
for ev in events.values():
    parts = [describe(x) for x in ev['new']]; rn = sorted({p[0] for p in parts if p[0] is not None})
    infl.append({'opp': ev['opp'], 'rounds': '+'.join(map(str, rn)), 'r0': rn[0], 'n': len(parts),
                 'kinds': '; '.join(k for _, k, _ in parts), 'quote': parts[-1][2], 'deltas': ev['pairs'],
                 'mean': float(np.mean(ev['pairs'])) if ev['pairs'] else np.nan,
                 'kept_before': ev['before'], 'kept_after': ev['after'], 'ruled_out': 100 * (1 - ev['after'] / ev['before'])})
infl = pd.DataFrame(infl).sort_values(['opp', 'r0', 'n']).reset_index(drop=True)
infl.drop(columns='deltas').to_csv(OUT / 'Figure2b_argument_influence.csv', index=False)
f4.to_csv(OUT / 'Figure2a_final_deals.csv', index=False)
infl[['opp', 'rounds', 'kinds', 'mean', 'ruled_out']]

## 5 — Figure 2: negotiation outcomes and argument influence

In [ ]:
def trim(s, n):
    return s if len(s) <= n else s[:n - 1].rstrip() + '…'

H_A = 3.2 + 0.72 * len(f4)                       # panel (a) grows with the number of distinct deals
fig = plt.figure(figsize=(14, H_A + 7.4))
sa, sb = fig.subfigures(2, 1, height_ratios=[H_A, 7.4], hspace=0.04)

# ---- (a) final deals vs NBS: distance from the NBS (efficiency-loss panel removed)
ax = sa.subplots()
y = np.arange(len(f4)); bh = 0.36
maxd = max(f4.euclid.max(), f4.manhattan.max(), 1)
be = ax.barh(y - bh / 2, f4.euclid, height=bh, color=GREY, edgecolor='black', lw=1.2)
bm = ax.barh(y + bh / 2, f4.manhattan, height=bh, color=CORAL, edgecolor='black', lw=1.2)
tr = blended_transform_factory(ax.transAxes, ax.transData)
for i, r in f4.iterrows():
    if r.nbs:
        ax.text(maxd * 0.015, i, '0  (the NBS itself)', va='center', fontsize=FS['ann'], fontweight='bold')
    else:
        for v, yy in [(r.euclid, i - bh / 2), (r.manhattan, i + bh / 2)]:
            ax.text(v + maxd * 0.012, yy, f'{v:.1f}'.rstrip('0').rstrip('.'), va='center', fontsize=FS['small'], fontweight='bold')
    ax.text(1.02, i, f'Hamming: {r.hamming}', transform=tr, va='center', fontsize=FS['ann'], fontstyle='italic', color='#333333')
    if not r.pareto:
        for b in (be[i], bm[i]): b.set_hatch('///')
    if not r.ir:
        be[i].set_alpha(0.45); bm[i].set_alpha(0.45)
ax.axvline(0, color='black', lw=2); ax.set_xlim(0, maxd * 1.15)
ax.set_xlabel('Distance from the NBS utility vector (utility points)')
ax.grid(axis='x', linestyle='--', alpha=0.6); ax.set_axisbelow(True); clean(ax, keep_left=False)
ax.set_yticks(y, [(f"{r.deal}{'  ★ NBS' if r.nbs else ''}\n{r['count']}× ({r.mods}) | {'Pareto' if r.pareto else 'not Pareto'}; {'IR' if r.ir else 'not IR'}"
                   if r.agreed else
                   f"NO AGREEMENT — rejected proposal {r.deal}\n{r['count']}× ({r.mods}) | blocked by {r.blockers or 'n/a'}")
                  for _, r in f4.iterrows()], fontsize=FS['tick'])
for lab, r in zip(ax.get_yticklabels(), f4.itertuples()):
    if not r.agreed: lab.set_color('#b03a2e')
for lab, r in zip(ax.get_yticklabels(), f4.itertuples()):
    if r.nbs: lab.set_fontweight('bold')
ax.tick_params(axis='y', length=0, pad=12)
sa.legend(handles=[mpatches.Patch(facecolor=GREY, edgecolor='black', label='Euclidean distance'),
                   mpatches.Patch(facecolor=CORAL, edgecolor='black', label='Manhattan distance'),
                   mpatches.Patch(facecolor='white', edgecolor='black', hatch='////', label='Not Pareto optimal'),
                   mpatches.Patch(facecolor=GREY, edgecolor='black', alpha=0.45, label='Faded: not individually rational')],
          loc='lower center', ncol=4, frameon=False, bbox_to_anchor=(0.55, -0.005), columnspacing=1.5)
sa.subplots_adjust(left=0.38, right=0.87, top=1 - 1.45 / H_A, bottom=1.35 / H_A)
by_mod = deals.assign(hit=(deals.deal == NBS) & deals.agreed).groupby('moderator').hit.agg(['sum', 'size'])
by_mod = by_mod.reindex([s for s in STAKEHOLDERS if s in by_mod.index])
letter(sa, 'a', 0.0, 0.995)
sa.text(0.045, 0.99, f'Final deals vs the NBS {NBS}', fontsize=FS['title'], fontweight='bold', va='top')
sa.text(0.045, 1 - 0.62 / H_A, f"NBS utilities {' / '.join(f'{SHORT[s]} {int(u)}' for s, u in zip(STAKEHOLDERS, U_nbs))} · Nash product {max_nash:,.0f} · "
        f"{n_nbs}/{n_runs} runs reached the NBS" + (f" · {n_noagree} ended without agreement" if n_noagree else ''),
        fontsize=FS['small'], va='top', color='#333333')
sa.text(0.045, 1 - 0.95 / H_A, 'NBS reached by moderator: ' + ' · '.join(f'{SHORT[m]} {int(r["sum"])}/{int(r["size"])}' for m, r in by_mod.iterrows()),
        fontsize=FS['small'], va='top', color='#333333')

# ---- (b) argument influence
df = infl.iloc[::-1].reset_index(drop=True); y = np.arange(len(df))
ax1, ax2 = sb.subplots(1, 2, sharey=True, gridspec_kw={'wspace': 0.1, 'width_ratios': [1, 0.8]})
vals = df['mean'].fillna(0).to_numpy()
bars = ax1.barh(y, vals, height=0.6, color=[BLUE if v >= 0 else CORAL for v in vals], edgecolor='black', lw=1.2)
for i, r in df.iterrows():
    if not r.deltas:
        bars[i].set_alpha(0); ax1.text(0.05, i, 'not re-evaluated', va='center', fontsize=FS['small'], fontstyle='italic', color='#333333'); continue
    for dlt in r.deltas: ax1.plot(dlt, i, 'o', ms=9, mfc='white', mec='black', mew=1.4, zorder=4)
    ax1.text(r['mean'] + (0.05 if r['mean'] >= 0 else -0.05), i + 0.33, f"{r['mean']:+.2f}", va='center',
             ha='left' if r['mean'] >= 0 else 'right', fontsize=FS['ann'], fontweight='bold')
ax1.axvline(0, color='black', lw=2); ax1.set_xlim(-1.05, 1.05)
ax1.set_title('Shift in P(accept)\nfor the same deal', fontsize=FS['title'])
ax2.barh(y, df.ruled_out, height=0.6, color=GREY, edgecolor='black', lw=1.2)
tr = blended_transform_factory(ax2.transAxes, ax2.transData)
for i, r in df.iterrows():
    ax2.text(r.ruled_out + 2, i, f'{r.ruled_out:.0f}%', va='center', fontsize=FS['ann'], fontweight='bold')
    kept = lambda v: f'{100 * v:.0f}%' if v >= 0.1 else f'{100 * v:.1f}%'
    ax2.text(1.02, i, f'kept {kept(r.kept_before)} → {kept(r.kept_after)}', transform=tr, va='center', fontsize=FS['small'], fontstyle='italic', color='#333333')
ax2.axvline(0, color='black', lw=2); ax2.set_xlim(0, 125)
ax2.set_title('Opponent model\nruled out (%)', fontsize=FS['title'])
for ax in (ax1, ax2):
    ax.grid(axis='x', linestyle='--', alpha=0.6); ax.set_axisbelow(True); clean(ax, keep_left=False)
ax1.set_yticks(y, [f"{SHORT[r.opp]}, round {r.rounds}: {trim(r.kinds, 34)}\n“{trim(r.quote, 44)}”" for r in df.itertuples()], fontsize=FS['tick'])
ax1.tick_params(axis='y', length=0, pad=12); ax2.tick_params(axis='y', left=False, labelleft=False)
sb.legend(handles=[mpatches.Patch(facecolor=BLUE, edgecolor='black', label='Raised P(accept)'),
                   mpatches.Patch(facecolor=CORAL, edgecolor='black', label='Lowered P(accept)'),
                   Line2D([0], [0], marker='o', ls='', mfc='white', mec='black', mew=1.4, ms=10, label='One deal before/after'),
                   mpatches.Patch(facecolor=GREY, edgecolor='black', label='Share ruled out')],
          loc='lower center', ncol=4, frameon=False, bbox_to_anchor=(0.55, -0.01), columnspacing=1.5)
sb.subplots_adjust(left=0.38, right=0.86, top=0.8, bottom=0.17)
letter(sb, 'b', 0.0, 0.99)
sb.text(0.045, 0.985, 'Influence of public statements on the opponent model (one traced negotiation)', fontsize=FS['title'], fontweight='bold', va='top')
save(fig, 'Figure2_negotiation_outcomes_and_argument_influence')
plt.show()